In [1]:
import torch
import timm
import joblib
import numpy as np
from torchvision import transforms
from PIL import Image

CLASS_NAMES = ["Flea Allergy", "Healthy", "Ringworm", "Scabies"]

# ── 1. Load DeiT ─────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deit = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=0)
deit.load_state_dict(torch.load("models/deit_feature_extractor.pth", map_location=device))
deit.to(device)
deit.eval()

# ── 2. Load PCA and SVM ──────────────────────────────────
pca = joblib.load("models/pca_transformer.pkl")
svm = joblib.load("models/rf_pca_model.pkl")

# ── 3. Preprocessing (same as training) ──────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── 4. Predict function ───────────────────────────────────
def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = deit(tensor).cpu().numpy()   # [1, 384]

    pca_features = pca.transform(embedding)      # [1, n_components]
    prediction   = svm.predict(pca_features)[0]

    print(f"Predicted class: {CLASS_NAMES[prediction]}")
    return CLASS_NAMES[prediction]

# ── 5. Test it ────────────────────────────────────────────
predict("image.png")

c:\Users\chanu\OneDrive - Informatics Institute of Technology\Desktop\IRP\Cat Skin DIsease Classification\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Predicted class: Healthy


'Healthy'